# Vendor Risk Intelligence — Pipeline Demo

This notebook walks through the full pipeline step by step.
Run cells sequentially. Set `BACKEND = 'vllm'` once your vLLM server is running on AMD.

In [ ]:
import sys, os
sys.path.insert(0, '..')   # project root

# ── Config ────────────────────────────────────────────────────────────────
TARGET_COMPANY = 'Apple Inc'
TARGET_TICKER  = 'AAPL'
BACKEND        = 'mock'   # Change to 'vllm' when AMD server is running

print(f'Target: {TARGET_COMPANY} | Backend: {BACKEND}')

## Step 1 — Generate Watchlist
Ask the LLM to map the supply chain for the target company.

In [ ]:
import asyncio
from src.models import PipelineState
from src.agents.watchlist_agent import watchlist_node

initial_state = PipelineState(
    target_company=TARGET_COMPANY,
    target_ticker=TARGET_TICKER,
    llm_backend=BACKEND,
    run_id='notebook-demo',
).model_dump()

state = await watchlist_node(initial_state)
ps    = PipelineState(**state)

print(f'Entities generated: {len(ps.entities)}')
print(f'Relationships:      {len(ps.relationships)}')
for e in ps.entities[:8]:
    print(f'  L{e.depth_level} | {e.entity_type.value:12s} | {e.name}')

## Step 2 — Collect Digital Footprints
Fan out to public APIs for each entity.

In [ ]:
from src.agents.footprint_agent import footprint_node

state = await footprint_node(state)
ps    = PipelineState(**state)

print(f'Footprints collected: {len(ps.footprint_data)}')
for eid, fp in list(ps.footprint_data.items())[:3]:
    fin = fp.financials
    print(f'\n  {fp.entity_name}')
    print(f'    News items:        {len(fp.news_items)}')
    print(f'    Negative news:     {fp.negative_news_count}')
    print(f'    Risk headlines:    {fp.risk_news_headlines[:2]}')
    if fin:
        print(f'    Market cap:        ${fin.market_cap_usd/1e9:.1f}B' if fin.market_cap_usd else '    Market cap: N/A')
        print(f'    Revenue growth:    {fin.revenue_growth_yoy_pct:.1f}%' if fin.revenue_growth_yoy_pct else '    Revenue growth: N/A')
        print(f'    Altman Z-Score:    {fin.altman_z_score}' if fin.altman_z_score else '    Altman Z-Score: N/A')

## Step 3 — Risk Scoring + Cascade Analysis

In [ ]:
from src.agents.risk_agent import risk_node

state = await risk_node(state)
ps    = PipelineState(**state)

# Show top risk entities
sorted_scores = sorted(ps.risk_scores.values(), key=lambda s: s.composite_score, reverse=True)

print(f'{'Entity':<30} {'Score':>6}  {'Level':<10}  Financial  Operational  Compliance  Geo')
print('-' * 90)
for s in sorted_scores[:10]:
    print(f'{s.entity_name[:30]:<30} {s.composite_score:>6.1f}  {s.risk_level.value:<10}  '
          f'{s.financial.score:>9.1f}  {s.operational.score:>11.1f}  '
          f'{s.compliance.score:>10.1f}  {s.geopolitical.score:>3.1f}')

print(f'\nAlerts generated: {len(ps.alerts)}')
for a in ps.alerts[:3]:
    print(f'  [{a.severity.value.upper()}] {a.alert_title}')

## Step 4 — Graph Metrics & Cascade Risk

In [ ]:
if ps.graph_metrics:
    gm = ps.graph_metrics
    print(f'Graph: {gm.total_nodes} nodes, {gm.total_edges} edges, density={gm.density:.3f}')
    print(f'Single points of failure: {len(gm.single_points_of_failure)}')
    print(f'\nTop cascade risk nodes:')
    for eid in gm.top_cascade_risks:
        nm     = gm.node_metrics.get(eid)
        entity = ps.entity_by_id(eid)
        name   = entity.name if entity else eid
        print(f'  {name:<30} blast_radius={nm.blast_radius_pct:.0f}%  '
              f'centrality={nm.betweenness_centrality:.3f}  '
              f'cascade_score={nm.cascade_risk_score:.1f}')

## Step 5 — Generate Executive Report

In [ ]:
from src.agents.report_agent import report_node

state = await report_node(state)
ps    = PipelineState(**state)
print(ps.report_html[:1000], '...')

## Step 6 — Generate HTML Dashboard

In [ ]:
from src.dashboard.html_generator import generate_dashboard_html, save_dashboard
from pathlib import Path

html = generate_dashboard_html(ps)
out  = Path('../data/outputs/demo_dashboard.html')
save_dashboard(html, out)

print(f'Dashboard: {out.resolve()}')
print(f'Size: {len(html):,} bytes')
print('Open the file in your browser to view the interactive dashboard.')

## Or — Run the Full Pipeline in One Call

In [ ]:
from src.pipeline.workflow import run_pipeline

full_state = await run_pipeline(
    target_company=TARGET_COMPANY,
    target_ticker=TARGET_TICKER,
    llm_backend=BACKEND,
)
print(f'Complete. Dashboard at: data/outputs/')
print(f'Entities: {len(full_state.entities)} | Alerts: {len(full_state.alerts)} | Errors: {len(full_state.errors)}')